In [1]:
import logging
import mlflow

from utils.data_prep import get_clean_combined_data
from model.train_baseline import train_evaluate_model

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

mlflow.sklearn.autolog(disable=True)

DOWNLOAD = False
REMOVE_ABYEI = False

Train: Data from January 2018 to December 2022. This includes the 2018 Sudanese revolution but excludes the 2023 Civil War.
Test onset civil war: Data from January 2023 to December 2023 which includes the escalation of the civil war.
Test active civil war: Data from January 2024 to December 2025 which includes fluctuations in ongoing civil war.


In [ ]:
ks = [0.5, 0.75]  # 0.25, 0.5,0.75,1
ns = [4]  # 4,5
event_cols = ["sub_event_type", "event_type"]

remove_abyei_options = [False]
include_food_options = [True, False]
include_rain_options = [True, False]

for remove_abyei in remove_abyei_options:
    for include_food in include_food_options:
        for include_rain in include_rain_options:
            data_sources = []
            if include_food:
                data_sources.append("food")
            if include_rain:
                data_sources.append("rain")

            abyei_str = "_remove_abyei" if remove_abyei else ""
            food_str = "_food" if include_food else ""
            rain_str = "_rain" if include_rain else ""
            which_data = f"acled{food_str}{rain_str}{abyei_str}"

            for k in ks:
                for n in ns:
                    for event_col in event_cols:
                        all_params = {
                            "max_depth": [3, 5, 7],
                            "min_child_weight": [1, 3, 5],
                            "max_delta_step": [0, 1, 5],
                            "gamma": [0, 1, 3, 5],
                            "learning_rate": [0.01, 0.03, 0.05, 0.1],
                            "subsample": [0.6, 0.8, 1.0],
                            "colsample_bytree": [0.6, 0.8, 1.0],
                            "reg_alpha": [0, 0.1, 1, 2],
                            "reg_lambda": [1, 5, 10],
                            "colsample_bylevel": [0.6, 0.8, 1.0],
                            "k": k,
                            "event_col": event_col,
                            "n_splits": n,
                            "remove_abyei": remove_abyei,
                        }

                        model_data, predictor_cols = get_clean_combined_data(
                            all_params,
                            data_sources=data_sources,
                            download=DOWNLOAD,
                            remove_abyei=remove_abyei,
                        )

                        event_str = (
                            "event"
                            if all_params["event_col"] == "event_type"
                            else "sub"
                        )
                        run_name = f"{which_data}_{all_params['k']}_{event_str}_{all_params['n_splits']}"

                        with mlflow.start_run(run_name=run_name):
                            mlflow.set_tag("data_version", which_data)
                            mlflow.set_tag("remove_abyei", remove_abyei)
                            mlflow.set_tag("include_food", include_food)
                            mlflow.set_tag("include_rain", include_rain)
                            mlflow.set_tag("k", all_params["k"])
                            mlflow.set_tag("n_splits", all_params["n_splits"])
                            mlflow.set_tag("event_col", all_params["event_col"])

                            results, best_params = train_evaluate_model(
                                model_data,
                                predictor_cols,
                                all_params,
                                best_params=False,
                            )

                            mlflow.log_params(best_params)
                            metrics_to_log = {
                                key: float(val) for key, val in results.items()
                            }
                            mlflow.log_metrics(metrics_to_log)
                            mlflow.log_dict(results, "model_report.json")

## Quick test

In [ ]:
data_sources = ["food", "rain"]  # Testing with all data sources
k = 0.5
n_splits = 4
event_col = "sub_event_type"

test_params = {
    "max_depth": [3],
    "min_child_weight": [1],
    "max_delta_step": [0],
    "gamma": [0],
    "learning_rate": [0.1],
    "subsample": [0.8],
    "colsample_bytree": [0.8],
    "reg_alpha": [0],
    "reg_lambda": [1],
    "colsample_bylevel": [0.8],
    "k": k,
    "event_col": event_col,
    "n_splits": n_splits,
    "remove_abyei": REMOVE_ABYEI,
}

print("Starting quick test run...")

try:
    print("Testing data loading...")
    model_data, predictor_cols = get_clean_combined_data(
        test_params,
        data_sources=data_sources,
        download=DOWNLOAD,
        remove_abyei=REMOVE_ABYEI,
    )
    print(f"Data loaded successfully. Shape: {model_data.shape}")

    run_name = "quick_test_run"

    print("Testing model training and MLflow logging...")
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("test_run", True)

        results, best_params = train_evaluate_model(
            model_data,
            predictor_cols,
            test_params,
            best_params=False,
        )

        mlflow.log_params(best_params)
        mlflow.log_metrics({key: float(val) for key, val in results.items()})
        mlflow.log_dict(results, "model_report.json")

    print("Yay")

except Exception as e:
    print(f"Error during test run: {e}")